# 사진 인증 모델 학습 (Travel Mission)

미션 2단계 **사진 인증**용 온디바이스 분류 모델을 학습한다.

- Colab: 런타임 → 런타임 유형 변경 → **GPU**
- `ml/labels.json` 을 이 노트북과 같은 위치에 두거나 저장소를 클론한다.
- 파이프라인: 데이터 로드 → CLIP 제로샷 베이스라인 → 헤드 학습 → 전체 파인튜닝 → 평가 → 임계값 선정 → HF Hub 업로드
- 산출물: `outputs/final/` (모델), `thresholds.json` (임계값)


In [ ]:
!pip install -q "transformers>=4.44" "datasets>=2.20" accelerate evaluate scikit-learn matplotlib peft torchvision

In [ ]:
import json
from pathlib import Path
import numpy as np
import torch

# ml/labels.json (앱과 공유하는 클래스 정의)
cfg = json.loads(Path("labels.json").read_text(encoding="utf-8"))
LABELS = cfg["labels"]
INVALID_LABEL = cfg["invalid_label"]
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}

BASE_MODEL = "apple/mobilevit-small"
DATASET_ID = "<user>/travel-mission-photos"   # HF Hub 비공개 데이터셋 (dataset_card.md 참고)
OUT_DIR = Path("outputs"); OUT_DIR.mkdir(exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE, LABELS)

## 1. 데이터셋 로드

`imagefolder` 규칙(`train/<라벨>/*.jpg`)으로 만든 HF Hub 데이터셋을 불러오고,
라벨 인덱스를 `labels.json` 순서에 맞춘다.

In [ ]:
from datasets import load_dataset

ds = load_dataset(DATASET_ID)   # train / validation / test

# imagefolder 는 라벨을 알파벳순으로 매기므로 labels.json 순서로 다시 매핑
hf_names = ds["train"].features["label"].names
remap = {i: label2id[name] for i, name in enumerate(hf_names)}
ds = ds.map(lambda b: {"label": [remap[x] for x in b["label"]]}, batched=True)

for split in ds:
    counts = np.bincount(ds[split]["label"], minlength=len(LABELS))
    print(split, dict(zip(LABELS, counts.tolist())))

## 2. CLIP 제로샷 베이스라인 (학습 없음)

`labels.json` 의 `clip_prompts` 로 텍스트 프롬프트를 만들고, 라벨별 최대 유사도로 분류한다.
보고서에 "학습 전" 기준선으로만 쓴다.

In [ ]:
from transformers import CLIPModel, CLIPProcessor
from sklearn.metrics import classification_report

CLIP_ID = "openai/clip-vit-base-patch32"   # 한국어 프롬프트를 쓰려면 Bingsu/clip-vit-large-patch14-ko
clip = CLIPModel.from_pretrained(CLIP_ID).to(DEVICE).eval()
clip_proc = CLIPProcessor.from_pretrained(CLIP_ID)

flat_prompts, owner = [], []
for lbl in LABELS:
    for p in cfg["clip_prompts"][lbl]:
        flat_prompts.append(p); owner.append(label2id[lbl])
owner = np.array(owner)

@torch.no_grad()
def clip_predict(images):
    inp = clip_proc(text=flat_prompts, images=list(images), return_tensors="pt", padding=True).to(DEVICE)
    sim = clip(**inp).logits_per_image.softmax(dim=-1).cpu().numpy()   # (N, P)
    per_label = np.stack([sim[:, owner == i].max(axis=1) for i in range(len(LABELS))], axis=1)
    return per_label.argmax(axis=1)

test_imgs = [im.convert("RGB") for im in ds["test"]["image"]]
test_y = np.array(ds["test"]["label"])
clip_pred = np.concatenate([clip_predict(test_imgs[i:i+32]) for i in range(0, len(test_imgs), 32)])
print(classification_report(test_y, clip_pred, target_names=LABELS, digits=3))

## 3. 전처리

`AutoImageProcessor` 의 리사이즈 크기·정규화 상수를 그대로 따른다.
**이 상수는 `export_onnx.py` 가 뽑아 주는 `preprocessor_config.json` 및 Android `OnnxPhotoVerifier` 전처리와 반드시 일치해야 한다.**

In [ ]:
from transformers import AutoImageProcessor
from torchvision.transforms import (Compose, RandomResizedCrop, Resize, CenterCrop,
                                    RandomHorizontalFlip, ToTensor, Normalize)

proc = AutoImageProcessor.from_pretrained(BASE_MODEL)
size = proc.size.get("shortest_edge") or proc.size.get("height") or 256
do_norm = getattr(proc, "do_normalize", False)
mean = getattr(proc, "image_mean", None) or [0.5, 0.5, 0.5]
std = getattr(proc, "image_std", None) or [0.5, 0.5, 0.5]
norm = [Normalize(mean, std)] if do_norm else []
print("size", size, "do_normalize", do_norm)

train_tf = Compose([RandomResizedCrop(size, scale=(0.7, 1.0)), RandomHorizontalFlip(), ToTensor(), *norm])
eval_tf = Compose([Resize(size), CenterCrop(size), ToTensor(), *norm])

def _apply(tf):
    def f(batch):
        batch["pixel_values"] = [tf(img.convert("RGB")) for img in batch["image"]]
        return batch
    return f

train_ds = ds["train"].with_transform(_apply(train_tf))
val_ds = ds["validation"].with_transform(_apply(eval_tf))
test_ds = ds["test"].with_transform(_apply(eval_tf))

## 4. 헤드 학습 (백본 freeze)

베이스라인. 분류 헤드만 학습하므로 데이터가 적어도 안정적이다.

In [ ]:
import evaluate
from transformers import AutoModelForImageClassification, TrainingArguments, Trainer

f1_metric = evaluate.load("f1")

def make_model():
    return AutoModelForImageClassification.from_pretrained(
        BASE_MODEL, num_labels=len(LABELS), id2label=id2label, label2id=label2id,
        ignore_mismatched_sizes=True)

def freeze_backbone(model):
    for name, p in model.named_parameters():
        if "classifier" not in name:
            p.requires_grad = False
    return model

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"macro_f1": f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"]}

def collate(examples):
    return {
        "pixel_values": torch.stack([e["pixel_values"] for e in examples]),
        "labels": torch.tensor([e["label"] for e in examples]),
    }

def run_training(model, tag, epochs, lr):
    args = TrainingArguments(
        output_dir=str(OUT_DIR / tag),
        per_device_train_batch_size=32, per_device_eval_batch_size=64,
        learning_rate=lr, num_train_epochs=epochs,
        eval_strategy="epoch", save_strategy="epoch",
        load_best_model_at_end=True, metric_for_best_model="macro_f1",
        logging_steps=20, fp16=torch.cuda.is_available(), report_to=[])
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                      data_collator=collate, compute_metrics=compute_metrics)
    trainer.train()
    return trainer

head_trainer = run_training(freeze_backbone(make_model()), "head", epochs=8, lr=1e-3)

## 5. 전체 파인튜닝

백본까지 학습. "개인화"의 핵심 실험이며, 4단계 결과와 표로 비교한다.

In [ ]:
full_trainer = run_training(make_model(), "full", epochs=6, lr=5e-5)

## (선택) LoRA 파라미터 효율적 파인튜닝

`target_modules` 이름은 `[n for n, _ in make_model().named_modules()]` 로 확인해 조정한다.

In [ ]:
# from peft import LoraConfig, get_peft_model
# lora_cfg = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05,
#                       target_modules=["query", "key", "value", "proj"],
#                       modules_to_save=["classifier"])
# lora_trainer = run_training(get_peft_model(make_model(), lora_cfg), "lora", epochs=8, lr=3e-4)

## 6. 평가 — classification report + confusion matrix

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

def evaluate_model(trainer, name):
    out = trainer.predict(test_ds)
    probs = torch.softmax(torch.tensor(out.predictions), dim=1).numpy()
    y = out.label_ids
    pred = probs.argmax(1)
    print(f"=== {name} ===")
    print(classification_report(y, pred, target_names=LABELS, digits=3))
    ConfusionMatrixDisplay(confusion_matrix(y, pred), display_labels=LABELS).plot(xticks_rotation=45)
    plt.title(name); plt.tight_layout(); plt.show()
    return probs, y

probs, y = evaluate_model(full_trainer, "full fine-tune")

## 7. 임계값 선정

각 사진에 대해
- `matchScore` = 정답 카테고리 확률
- `invalidScore` = `무효` 확률

를 보고 앱 `PhotoVerificationConfig` 의 세 임계값을 정한다.
목표: **무효 재현율 ≳ 0.9**, **정상 사진 오탐율 최소화**.

In [ ]:
invalid_id = label2id[INVALID_LABEL]
is_invalid = (y == invalid_id)
inv_scores = probs[:, invalid_id]

print("[invalidRejectThreshold 후보]")
for t in np.round(np.arange(0.30, 0.91, 0.05), 2):
    rec = (inv_scores[is_invalid] >= t).mean()
    fp = (inv_scores[~is_invalid] >= t).mean()
    print(f"  t={t:.2f}  무효recall={rec:.3f}  정상오탐={fp:.3f}")

valid = ~is_invalid
match = probs[np.arange(len(y)), y][valid]
print("[autoPass / hardReject 후보 — 정상 사진의 정답 카테고리 확률]")
for t in np.round(np.arange(0.30, 0.91, 0.05), 2):
    print(f"  t={t:.2f}  이상 비율={(match >= t).mean():.3f}")

# --- 위 표를 보고 아래 3개 값을 채운다 ---
AUTO_PASS = 0.70
HARD_REJECT = 0.30
INVALID_REJECT = 0.60

from sklearn.metrics import f1_score
result = {
    "model_version": "mobilevit-small-fullft-0",
    "autoPassThreshold": AUTO_PASS,
    "hardRejectThreshold": HARD_REJECT,
    "invalidRejectThreshold": INVALID_REJECT,
    "eval": {
        "invalid_recall": float((inv_scores[is_invalid] >= INVALID_REJECT).mean()),
        "valid_false_reject_rate": float((match < HARD_REJECT).mean()),
        "macro_f1": float(f1_score(y, probs.argmax(1), average="macro")),
    },
}
Path("thresholds.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(result, ensure_ascii=False, indent=2))

## 8. 모델 저장 · HF Hub 업로드

In [ ]:
full_trainer.save_model(str(OUT_DIR / "final"))
proc.save_pretrained(str(OUT_DIR / "final"))

# from huggingface_hub import notebook_login
# notebook_login()
# full_trainer.model.push_to_hub("<user>/travel-mission-photo-verifier")
# proc.push_to_hub("<user>/travel-mission-photo-verifier")

## 다음 단계

```bash
python export_onnx.py --model outputs/final \
    --out ../app/src/main/assets/photo_verifier.onnx --quantize
```

1. `thresholds.json` 값을 앱 `PhotoVerificationConfig` 기본값으로 옮긴다.
2. `photo_verifier_preprocessor.json` 의 `size` / `image_mean` / `image_std` / `do_normalize` 를
   Android `OnnxPhotoVerifier` 전처리와 일치시킨다.
3. `photo_verifier_labels.json` 의 라벨 순서로 앱이 출력 인덱스를 읽는다.
